# MLIP Active-Learning Tutorial

> New to ALF? Start with the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

This tutorial shows how the ALF `MLIPModel` (a MACE machine-learned interatomic
potential) is used in an **offline (pool-based) active-learning loop**. Our use case: starting from
a pretrained foundation model and **finetuning** it into an accurate force field for a single
organic molecule, while spending as few expensive labels as possible.

### How this differs from the design tutorials

The protein-design tutorials *maximise a fitness*. Here we do the opposite kind of active
learning: we **minimise model error using as few expensive labels as possible**. Each label is an
expensive quantum-chemistry calculation (DFT). We work from a fixed pool of DFT-labelled aspirin
configurations and let the model decide which ones are worth "paying" to reveal — choosing the
configurations where its committee of models *disagrees most* (often the most informative ones,
though, as we'll see, not always).

### Experiment overview

1. Download a pretrained MACE organics model and a pool of DFT-labelled aspirin configurations.
2. Finetune a committee (ensemble) of models on a small seed set and use their disagreement to
   estimate prediction uncertainty.
3. Each round: score the remaining candidate pool, acquire the most uncertain configurations,
   reveal their DFT labels, and finetune again.
4. Compare an uncertainty-driven acquisition against a random baseline, and see why acquisition
   choice matters.

### Framework Components

1. **Dataset** (`AspirinDataset`, defined below): loads DFT-labelled aspirin configurations and
   splits them into seed-train / validation / test / **candidate pool**. The pool holds the
   unlabelled configurations active learning chooses from.
2. **Surrogate Model** ([`MLIPModel`](https://instadeepai.github.io/alf/api/alf_tools/models/)) wrapped in an [`EnsembleWrapper`](https://instadeepai.github.io/alf/api/alf_tools/models/) committee: each member finetunes the pretrained MACE model; their disagreement is our uncertainty.
3. **Search Strategy** ([`DatasetSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/)): serves the remaining candidate pool each round.
4. **Acquisition Function** ([`UncertaintyBased`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/) from `alf_tools`): selects the configurations where the committee disagrees most (highest prediction variance). [`RandomSelection`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/) is the baseline for comparison.
5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): handles the ask/tell cycle.
6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/) with the dataset as scorer): reveals the precomputed DFT energy (and forces) for an acquired configuration via `dataset.query`.
7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): orchestrates the active-learning loop.

## Setup

These tutorials are written for **dev mode** — running from a local clone of the ALF repository.
The MLIP tutorial needs the `mlip` package (the MACE force field), which ALF exposes through the
optional `mlip` extra (mirrored as the `mlip` dependency group in `tutorials/pyproject.toml`). From
the `tutorials/` directory, sync that group so the extra is installed alongside the tutorial
dependencies:

```bash
uv sync --group mlip   # installs alf_core, alf_tools[mlip] and tutorial deps (CPU PyTorch by default)
```

Register the environment as a Jupyter kernel, then select the `alf` kernel in this notebook:

```bash
uv run ipython kernel install --user --env VIRTUAL_ENV "$(pwd)/.venv" --name=alf
```

`uv sync` installs the **CPU** build of PyTorch by default. For GPU acceleration, see the
[GPU support section of the Installation Guide](https://instadeepai.github.io/alf/installation.html#gpu-support).

**Not running from a clone?** If you installed ALF with `pip`, run the optional cell below to
install this tutorial's dependencies into the current kernel, then restart the kernel.

In [ ]:
# Optional — only needed if you are NOT running from a cloned repo via `uv sync`.
# Installs ALF and this tutorial's dependencies into the current kernel, then restart the kernel.
# %pip install "alf_tools[mlip]" matplotlib pandas huggingface_hub

### Step 0: Download the Model and Dataset

The cell below downloads the pretrained MACE organics model and the public **rMD17 aspirin** dataset
(both from Hugging Face, public, no credentials) into the local cache. The aspirin configurations
carry DFT energies and forces, so no live quantum-chemistry engine is needed.

In [ ]:
from pathlib import Path

# Pretrained MACE organics foundation model -> alf models dir, so the MLIPModel loader
# finds it locally and skips its (private) S3 fallback.
import alf_tools.models.utils.mlip_utils as mlip_utils  # noqa: E402
from huggingface_hub import hf_hub_download, snapshot_download

models_dir = mlip_utils._MODELS_DIR
models_dir.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="mace_organics_02.zip",
    local_dir=str(models_dir),
)

# Public rMD17 aspirin dataset (DFT energies + forces) from the mlip tutorials collection.
DATA_DIR = Path("data/aspirin")
snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="training/rmd17_aspirin_*",
    local_dir=str(DATA_DIR),
)
ASPIRIN_TRAIN_XYZ = DATA_DIR / "training" / "rmd17_aspirin_train.xyz"

print(f"✅ Pretrained model present at {models_dir / 'mace_organics_02.zip'}")
print(f"✅ rMD17 aspirin pool present at {ASPIRIN_TRAIN_XYZ}")

### Step 1: Import Required Libraries

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from alf_core import (
    Candidate,
    DatasetSearch,
    DesignTask,
    FileStateLogger,
    LabelledCandidates,
    Optimizer,
    Oracle,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig, SubsampleConfig
from alf_tools.models.mlip import MLIPModel, MLIPModelConfig, MLIPTrainConfig
from alf_tools.optimizer.acquisition_functions import UCB, RandomSelection
from ase.io import read as ase_read

print("✅ All imports successful!")

### Step 2: Load the rMD17 Aspirin Dataset (the molecule pool)

We load configurations of **aspirin** from the revised MD17 (rMD17) dataset — these are the pool
of candidate molecules we optimise over. Our objective is **stability**, defined as `-energy` (eV):
the most stable conformer is the lowest-energy one, and ALF maximises, so we store `-energy` as the
label. We subsample `N_POOL` configurations and let ALF split them into a small **seed-train** set,
a **validation** set (the surrogate needs it for finetuning), and a **candidate pool** to search.

In [ ]:
DATA_SEED = 51505
N_POOL = 150  # subsample of rMD17 aspirin configs (the molecule pool to optimise over)


def load_aspirin(xyz_path: Path | str, max_configs: int | None = None, seed: int = 0) -> LabelledCandidates:
    """Read aspirin configurations into `LabelledCandidates`.

    The optimisation objective is **stability**, defined as `-energy` (eV): higher stability
    means a lower-energy, more stable conformer. ALF maximises labels, so storing stability lets
    `UCB`, `get_top_k`, and the recall/regret metrics all treat "higher = better".
    """
    atoms_list = ase_read(str(xyz_path), index=":")
    if max_configs is not None and len(atoms_list) > max_configs:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(atoms_list), size=max_configs, replace=False)
        atoms_list = [atoms_list[i] for i in idx]
    candidates = [Candidate(data=atoms, modality=Modality.STRUCTURE) for atoms in atoms_list]
    stability = np.asarray([-atoms.get_potential_energy() for atoms in atoms_list])
    return LabelledCandidates(candidates=candidates, labels=stability)


class AspirinDataset(BaseDataset):
    """Pre-labelled aspirin conformers (label = stability = -energy), split into seed / val / pool."""

    def __init__(self, config: BaseDatasetConfig, data: LabelledCandidates):
        """Store the pre-built labelled data; `setup()` performs the split."""
        super().__init__(config)
        self._data = data

    def load_dataset(self) -> LabelledCandidates:
        """Return the pre-built `LabelledCandidates` (stability labels)."""
        return self._data


def make_aspirin_dataset(data: LabelledCandidates | None = None, seed: int = DATA_SEED) -> AspirinDataset:
    """Build and set up an `AspirinDataset` with the CPU-tiny split used in this tutorial."""
    if data is None:
        data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_POOL, seed=seed)
    ds = AspirinDataset(
        BaseDatasetConfig(
            name="rmd17_aspirin",
            modality=Modality.STRUCTURE,
            seed=seed,
            train_ratio=0.12,
            validation_frac=0.2,
            test_ratio=0.0,  # Bayesian optimisation tracks the best in the pool; no held-out test
            split_type="random",
            problem_type=ProblemType.REGRESSION,
        ),
        data,
    )
    ds.setup()
    return ds


data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_POOL, seed=DATA_SEED)

# Fail fast if aspirin contains any element the pretrained organics model never saw.
from mlip.models.mace.network import Mace  # noqa: E402

_pretrained_ff = mlip_utils._load_model_from_zip(Mace, "mace_organics_02.zip")
_ztable = set(_pretrained_ff.dataset_info.atomic_energies_map.keys())
_missing = {int(n) for c in data.candidates for n in c.data.numbers} - _ztable
if _missing:
    raise ValueError(f"aspirin contains elements outside the pretrained z-table: {_missing}")

dataset = make_aspirin_dataset(data)
print(dataset)
print(
    f"✅ rMD17 aspirin pool ready — seed-train={len(dataset.train_dataset)}, "
    f"pool={len(dataset.candidate_pool)} (objective: maximise stability = -energy)"
)

### Step 3: The Oracle (precomputed DFT)

The oracle is the expensive ground-truth evaluator. Here each aspirin configuration already has a
DFT energy, so the oracle simply **reveals** that label on demand: `Oracle(scorer=dataset)` calls
`dataset.query(...)` to look up the energy for an acquired configuration. This mirrors production
active learning, where labelling is costly and is therefore spent sparingly on the most informative
structures.

> **Offline vs online.** This is the **offline** setting: the oracle matches acquired candidates to
> their labels *by object identity* against the dataset, so it only works for configurations already
> in the pool. If instead you want to **generate new structures on the fly** and label them with a
> live evaluator (a running DFT/xTB calculation, or a pretrained MLIP used *as* the oracle), use a
> `BaseModel`-backed oracle and a generative search — see the
> [online design tutorial](https://github.com/instadeepai/alf/blob/main/tutorials/experiments/online_design_tutorial.ipynb).

In [ ]:
# Offline oracle: reveal the precomputed DFT label for an acquired configuration.
# In production this is an expensive DFT calculation; here the labels already exist in
# the dataset, so the oracle is a lookup (`dataset.query`) keyed by candidate identity.
oracle = Oracle(scorer=dataset)
print("✅ Oracle ready (offline lookup of precomputed DFT labels)!")

### Step 4: Search and Acquisition

`DatasetSearch` serves the remaining **candidate pool**. `UCB` (Upper Confidence Bound) scores each
candidate by `μ + α·σ` — the committee's mean predicted stability plus `α` times its uncertainty —
so it balances **exploiting** molecules predicted to be very stable against **exploring** ones the
committee is unsure about. `RandomSelection` is the baseline that picks at random, to show the
surrogate-guided search actually finds the best molecule faster.

In [ ]:
# DatasetSearch serves the remaining candidate pool to the acquisition each round.
search_fn = DatasetSearch()
print("✅ Search ready (offline candidate pool)!")

### Step 5: The Surrogate — a Committee of Finetuned MACE Models

Our surrogate is an ensemble (committee) of `MLIPModel`s, each finetuning the **same** pretrained
MACE model on a different bootstrap resample of the labelled set, so they disagree where data is
scarce. `EnsembleWrapper.predict` returns the mean predicted **stability** and the variance across
members — both consumed by `UCB`.

*Note:* the surrogate predicts the stability label (`-energy`); MACE re-fits a per-element
reference energy from the training data. For a single molecule (constant composition) that fit is
degenerate but harmless — it subtracts the correct constant — so stability is reproduced on a
consistent scale.

In [ ]:
def mlip_factory(seed: int) -> MLIPModel:
    """Build a finetuning MLIPModel (from the pretrained MACE model) with the given seed."""
    return MLIPModel(
        model_config=MLIPModelConfig(model_path="mace_organics_02.zip"),
        train_config=MLIPTrainConfig(epochs=8, batch_size=2, learning_rate=1e-3),
        seed=seed,
    )


def make_surrogate(n_members: int = 2) -> Surrogate:
    """Build a committee surrogate of finetuned MACE models (1 member = no uncertainty)."""
    return Surrogate(
        model=EnsembleWrapper(
            model_factory=mlip_factory,
            config=EnsembleWrapperConfig(
                base_seed=0,
                n_members=n_members,
                subsample=SubsampleConfig(fraction=1.0, replace=True),
            ),
        )
    )


surrogate = make_surrogate(n_members=2)
print("✅ Surrogate committee initialised (2 finetuned MACE members)!")

### Step 6: Optimizer and Design Task

`UncertaintyBased` ranks candidates purely by the committee's prediction variance, so each round we
label the pool configurations the ensemble disagrees on most — classic query-by-committee active
learning. The `Optimizer` combines it with the `DatasetSearch` over the candidate pool; `DesignTask`
runs the rounds.

In [ ]:
acquisition_fn = UCB(alpha=1.0)  # mu + alpha*sigma over predicted stability; raise alpha to explore more
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

num_acq_rounds = 3
acq_batch_size = 3
task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)
print(f"✅ Optimizer + DesignTask ready ({num_acq_rounds} rounds × {acq_batch_size} labels)!")

### Step 7: Run the Bayesian-Optimisation Loop (UCB)

Each round: score the remaining pool with the committee, acquire the highest-`UCB` molecules,
reveal their true DFT stability via the oracle, finetune the committee, and record progress.

> **⏱️ Slowest cell.** Finetunes a 2-member MACE committee each round; expect a few minutes on CPU.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING)

ucb_path = Path("results/mlip_design/")
if ucb_path.exists():
    shutil.rmtree(ucb_path)
loggers = [TerminalStateLogger(), FileStateLogger(output_path=ucb_path)]

state = task.setup(dataset=dataset, surrogate=surrogate)
print("🚀 Running UCB Bayesian optimisation...")
task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)
print("✅ UCB experiment completed!")

### Step 8: Random-Selection Baseline

To isolate the effect of the *acquisition strategy*, the baseline runs the identical loop on the
same aspirin splits with the **same 2-member committee surrogate** — the only thing that changes is
*how* molecules are chosen (uniformly at random instead of by UCB score). Keeping the surrogate
fixed makes the two curves a fair, apples-to-apples comparison.

> **⏱️ Slow cell** (comparable to Step 7): another committee finetuned across acquisition rounds.

In [ ]:
# Rebuild a fresh dataset so the baseline starts from the same initial splits.
dataset_random = make_aspirin_dataset()
random_oracle = Oracle(scorer=dataset_random)  # oracle is bound to THIS dataset's labels

# Same 2-member committee as the UncertaintyBased run — only the acquisition differs.
random_optimizer = Optimizer(acquisition_fn=RandomSelection(seed=0), search_fn=search_fn)
random_surrogate = make_surrogate(n_members=2)
random_task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)

random_path = Path("results/mlip_design_random/")
if random_path.exists():
    shutil.rmtree(random_path)
random_loggers = [TerminalStateLogger(), FileStateLogger(output_path=random_path)]

random_state = random_task.setup(dataset=dataset_random, surrogate=random_surrogate)
print("🚀 Running random-selection baseline...")
random_task.run(
    random_state, state_loggers=random_loggers, optimizer=random_optimizer, oracle=random_oracle
)
print("✅ Baseline completed!")

### Step 9: Results — did the surrogate find the best molecule faster?

The headline metric is **regret**: the stability gap between the best molecule *acquired so far* and
the best molecule *in the whole pool* (0 = the most stable conformer has been found; lower is
better). We also show **top-10% recall** (what fraction of the pool's most-stable molecules have
been acquired). If the UCB-guided surrogate works, its regret drops to 0 faster than random.

In [ ]:
ucb_metrics = pd.read_csv("results/mlip_design/metrics.csv")
rnd_metrics = pd.read_csv("results/mlip_design_random/metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MLIP Bayesian optimisation: UCB vs random", fontsize=15, fontweight="bold")
for df, label, color in [
    (ucb_metrics, "UCB (committee)", "#e74c3c"),
    (rnd_metrics, "Random", "#3498db"),
]:
    d = df.dropna(subset=["optimizer/regret"])  # regret/recall start at round 1
    axes[0].plot(d["dataset/num_train"], d["optimizer/regret"], marker="o", label=label, color=color)
    axes[1].plot(
        d["dataset/num_train"], d["optimizer/top_10pc_recall"], marker="o", label=label, color=color
    )

axes[0].set_xlabel("Number of labelled molecules")
axes[0].set_ylabel("Regret (stability, eV)")
axes[0].set_title("Regret (lower is better; 0 = best found)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Number of labelled molecules")
axes[1].set_ylabel("Top-10% recall")
axes[1].set_title("Recall of the most-stable molecules (higher is better)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Best molecule found by the UCB run vs the best obtainable in the pool.
best_possible = float(dataset.init_candidate_pool.labels.max())
best_found = float(dataset.train_dataset.labels.max())  # seed + acquired
print(f"Best stability in pool      : {best_possible:.4f} eV  (energy {-best_possible:.4f} eV)")
print(f"Best found by UCB so far    : {best_found:.4f} eV  (energy {-best_found:.4f} eV)")

**Reading the results.** Regret falling to 0 means the loop has acquired the single most stable
conformer in the pool; the faster a curve reaches 0, the more sample-efficient the strategy. On a
tiny pool with few rounds the two strategies can be close (and random is a strong baseline), so
treat the curves as illustrative — the point is the end-to-end BO loop, and that the surrogate's
uncertainty (via UCB) gives it something to exploit. The printout reports the best molecule found
against the best obtainable in the pool.

## Conclusion

We built an offline (pool-based) active-learning loop for a machine-learned interatomic potential
with ALF: finetune a pretrained MACE committee on a few DFT-labelled aspirin configurations,
estimate uncertainty from committee disagreement, acquire more configurations from a candidate pool,
and repeat. Every component is a standard ALF abstraction — `AspirinDataset`, the dataset-lookup
`Oracle`, `DatasetSearch`, the `EnsembleWrapper` surrogate, `UncertaintyBased`/`RandomSelection`,
`Optimizer`, and `DesignTask` — and both acquisition strategies share the same committee surrogate
on a leakage-free (disjoint-trajectory) test set.

This example is deliberately tiny and fast, so treat the learning curves as illustrative rather
than conclusive. On a problem this small, uncertainty sampling does not reliably beat random
selection — a useful reminder that the acquisition strategy must be matched to the problem and
validated, not assumed. Query-by-committee active learning is the workhorse of *production* MLIP
training, where it selects from large, diverse pools of physically meaningful structures.

### Next steps

- **Scale up** so the uncertainty signal can show: more committee members, more acquisition rounds,
  a larger pool, and a larger held-out test set (rMD17 aspirin ships 1200/900/900 configs).
- **Try a stronger uncertainty estimate**: a deep-kernel / last-layer GP or a last-layer Laplace
  approximation on the MACE features gives calibrated epistemic variance (the latter is the
  NTK-flavoured, more principled cousin of an ensemble).
- **Go online**: generate candidates on the fly and label them with a live oracle (DFT/xTB, or a
  pretrained MLIP used *as* the oracle) — see the offline-vs-online note in Step 3.
- **Try a different molecule** or train from scratch (`model_path=None`) for chemistries with no
  compatible foundation model.

**Happy modelling!** ⚛️🔬✨

In [ ]:
for p in [Path("results/mlip_design/"), Path("results/mlip_design_random/")]:
    if p.exists():
        shutil.rmtree(p)
print("✅ Results directories cleaned up!")